In [1]:
!pip install -U python-jobspy
!pip install beautifulsoup4 
!pip install requests
!pip install pandas
!pip install numpy
!pip install openpyxl
!pip install xlrd
!pip install xlwt
!pip install xlutils

In [1]:
import sqlite3
from datetime import datetime, timedelta
from jobspy import scrape_jobs
import pandas as pd
import time
import logging
from typing import List, Dict, Any
import random

class JobScraperETL:
    def __init__(self, db_path: str = "jobs.db"):
        """Initialize the ETL pipeline with database connection."""
        self.db_path = db_path
        self.setup_logging()
        self.setup_database()
        
        # List of proxy servers - replace with your actual proxies
        self.proxies = [
            "localhost"  # Add your proxy servers here
        ]
        
        # Job search parameters
        self.search_locations = [
            "New York, NY", "San Francisco, CA", "Seattle, WA",
            "Austin, TX", "Boston, MA", "Chicago, IL"
        ]
        
        self.job_titles = [
            "software engineer", "software developer", 
            "data scientist", "machine learning engineer",
            "data engineer", "full stack developer"
        ]

    def setup_logging(self):
        """Configure logging for the ETL process."""
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler('job_scraper.log'),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)

    def setup_database(self):
        """Create SQLite database and tables if they don't exist."""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        # Create jobs table
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS jobs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            job_id TEXT UNIQUE,
            title TEXT,
            company TEXT,
            company_url TEXT,
            job_url TEXT,
            location_country TEXT,
            location_city TEXT,
            location_state TEXT,
            description TEXT,
            job_type TEXT,
            salary_interval TEXT,
            salary_min_amount REAL,
            salary_max_amount REAL,
            salary_currency TEXT,
            date_posted TIMESTAMP,
            is_remote BOOLEAN,
            job_function TEXT,
            company_industry TEXT,
            source_site TEXT,
            scrape_date TIMESTAMP,
            UNIQUE(job_url, company, title)
        )
        """)
        
        conn.commit()
        conn.close()

    def generate_job_id(self, row: Dict[str, Any]) -> str:
        """Generate a unique job ID based on job details."""
        components = [
            str(row.get('job_url', '')),
            str(row.get('company', '')),
            str(row.get('title', '')),
            str(row.get('location_city', '')),
            str(row.get('date_posted', ''))
        ]
        return '_'.join(components)

    def scrape_jobs_for_location(self, location: str, search_term: str) -> pd.DataFrame:
        """Scrape jobs for a specific location and search term."""
        try:
            self.logger.info(f"Scraping jobs for {search_term} in {location}")
            
            jobs = scrape_jobs(
                site_name=["indeed", "linkedin", "zip_recruiter", "glassdoor"],
                search_term=search_term,
                location=location,
                results_wanted=1000,  # Adjust based on your needs
                hours_old=72,
                country_indeed='USA',
                proxies=self.proxies,
                verbose=1
            )
            
            if jobs is not None and not jobs.empty:
                jobs['scrape_date'] = datetime.now()
                jobs['source_site'] = jobs['site']
                return jobs
            
            return pd.DataFrame()
            
        except Exception as e:
            self.logger.error(f"Error scraping jobs for {location}: {str(e)}")
            return pd.DataFrame()

    def transform_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """Transform and clean the scraped data."""
        if df.empty:
            return df
            
        # Generate unique job IDs
        df['job_id'] = df.apply(self.generate_job_id, axis=1)
        
        # Clean salary data
        df['salary_min_amount'] = pd.to_numeric(df['min_amount'], errors='coerce')
        df['salary_max_amount'] = pd.to_numeric(df['max_amount'], errors='coerce')
        df['salary_interval'] = df['interval']
        
        # Clean and standardize columns
        df['date_posted'] = pd.to_datetime(df['date_posted'], errors='coerce')
        df['scrape_date'] = pd.to_datetime(df['scrape_date'])
        
        # Select and rename columns for database
        columns_mapping = {
            'job_id': 'job_id',
            'title': 'title',
            'company': 'company',
            'company_url': 'company_url',
            'job_url': 'job_url',
            'country': 'location_country',
            'city': 'location_city',
            'state': 'location_state',
            'description': 'description',
            'job_type': 'job_type',
            'salary_interval': 'salary_interval',
            'salary_min_amount': 'salary_min_amount',
            'salary_max_amount': 'salary_max_amount',
            'currency': 'salary_currency',
            'date_posted': 'date_posted',
            'is_remote': 'is_remote',
            'job_function': 'job_function',
            'company_industry': 'company_industry',
            'source_site': 'source_site',
            'scrape_date': 'scrape_date'
        }
        
        return df.rename(columns=columns_mapping)[list(columns_mapping.values())]

    def load_to_database(self, df: pd.DataFrame):
        """Load transformed data into SQLite database."""
        if df.empty:
            self.logger.warning("No data to load into database")
            return
            
        try:
            conn = sqlite3.connect(self.db_path)
            
            # Insert new records, ignore duplicates
            df.to_sql('jobs', conn, if_exists='append', index=False, 
                     method='multi', chunksize=1000)
            
            conn.commit()
            self.logger.info(f"Successfully loaded {len(df)} jobs into database")
            
        except Exception as e:
            self.logger.error(f"Error loading data to database: {str(e)}")
            
        finally:
            conn.close()

    def run_etl_pipeline(self):
        """Execute the complete ETL pipeline."""
        total_jobs = 0
        
        for location in self.search_locations:
            for job_title in self.job_titles:
                try:
                    # Add random delay between searches to avoid rate limiting
                    time.sleep(random.uniform(2, 5))
                    
                    # Extract
                    raw_data = self.scrape_jobs_for_location(location, job_title)
                    
                    if not raw_data.empty:
                        # Transform
                        transformed_data = self.transform_data(raw_data)
                        
                        # Load
                        self.load_to_database(transformed_data)
                        
                        total_jobs += len(transformed_data)
                        
                except Exception as e:
                    self.logger.error(f"Pipeline error for {job_title} in {location}: {str(e)}")
                    continue
        
        self.logger.info(f"ETL pipeline completed. Total jobs processed: {total_jobs}")

if __name__ == "__main__":
    # Initialize and run the ETL pipeline
    etl = JobScraperETL()
    etl.run_etl_pipeline()

2025-01-31 23:33:52,226 - INFO - Scraping jobs for software engineer in New York, NY
2025-01-31 23:34:04,964 - ERROR - JobSpy:ZipRecruiter - 429 Response - Blocked by ZipRecruiter for too many requests
2025-01-31 23:35:24,361 - INFO - JobSpy:Linkedin - finished scraping
2025-01-31 23:35:24,629 - ERROR - Pipeline error for software engineer in New York, NY: "['location_country', 'location_city', 'location_state'] not in index"
2025-01-31 23:35:28,350 - INFO - Scraping jobs for software developer in New York, NY
2025-01-31 23:37:00,236 - ERROR - JobSpy:ZipRecruiter - 429 Response - Blocked by ZipRecruiter for too many requests
2025-01-31 23:37:00,659 - ERROR - Pipeline error for software developer in New York, NY: "['location_country', 'location_city', 'location_state'] not in index"
2025-01-31 23:37:03,647 - INFO - Scraping jobs for data scientist in New York, NY
2025-01-31 23:37:04,165 - ERROR - JobSpy:ZipRecruiter - 429 Response - Blocked by ZipRecruiter for too many requests
2025-01-

KeyboardInterrupt: 

In [ ]:
import sqlite3
from datetime import datetime, timedelta
from jobspy import scrape_jobs
import pandas as pd
import time
import logging
from typing import List, Dict, Any, Optional
import random
from tenacity import retry, stop_after_attempt, wait_exponential
import concurrent.futures
from dataclasses import dataclass
from queue import Queue
from threading import Lock

@dataclass
class RateLimiter:
    requests: int = 0
    window_start: float = time.time()
    lock: Lock = Lock()

class JobScraperETL:
    def __init__(self, db_path: str = "jobs.db"):
        """Initialize the ETL pipeline with database connections."""
        self.db_path = db_path
        self.setup_logging()
        self.setup_database()
        
        # Rate limiting settings
        self.rate_limiters = {
            'indeed': RateLimiter(),
            'linkedin': RateLimiter(),
            'zip_recruiter': RateLimiter(),
            'glassdoor': RateLimiter()
        }
        self.max_requests = 10  # requests per window
        self.window_size = 60   # seconds
        
        # List of proxy servers - replace with your actual proxies
        self.proxies = [
            "localhost"  # Add your proxy servers here
        ]
        
        # Job search parameters
        self.search_locations = [
            "New York, NY", "San Francisco, CA", "Seattle, WA",
            "Austin, TX", "Boston, MA", "Chicago, IL"
        ]
        
        self.job_titles = [
            "software engineer", "software developer", 
            "data scientist", "machine learning engineer",
            "data engineer", "full stack developer"
        ]
        
        # Queue for storing scraped jobs
        self.job_queue = Queue(maxsize=1000)
        
    def check_rate_limit(self, site: str) -> bool:
        """Check if we're within rate limits for a given site."""
        limiter = self.rate_limiters[site]
        with limiter.lock:
            current_time = time.time()
            if current_time - limiter.window_start > self.window_size:
                limiter.requests = 0
                limiter.window_start = current_time
            
            if limiter.requests >= self.max_requests:
                return False
            
            limiter.requests += 1
            return True

    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=4, max=10))
    def scrape_jobs_for_location(self, location: str, search_term: str, site: str) -> Optional[pd.DataFrame]:
        """Scrape jobs for a specific location and search term from a specific site."""
        try:
            if not self.check_rate_limit(site):
                self.logger.warning(f"Rate limit reached for {site}, waiting...")
                time.sleep(self.window_size)
            
            self.logger.info(f"Scraping jobs for {search_term} in {location} from {site}")
            
            jobs = scrape_jobs(
                site_name=[site],
                search_term=search_term,
                location=location,
                results_wanted=1000,
                hours_old=72,
                country_indeed='USA',
                linkedin_fetch_description=False,
                proxies=self.proxies,
                verbose=1
            )
            
            if jobs is not None and not jobs.empty:
                jobs['scrape_date'] = datetime.now()
                jobs['source_site'] = site
                return jobs
            
            return None
            
        except Exception as e:
            self.logger.error(f"Error scraping jobs from {site} for {location}: {str(e)}")
            raise

    def setup_logging(self):
        """Configure logging for the ETL process."""
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler('job_scraper.log'),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)

    def setup_database(self):
        """Create SQLite database and tables if they don't exist."""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        
        # Create jobs table for SQLite
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS jobs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            job_id TEXT UNIQUE,
            title TEXT,
            company TEXT,
            company_url TEXT,
            job_url TEXT,
            location_country TEXT,
            location_city TEXT,
            location_state TEXT,
            description TEXT,
            job_type TEXT,
            salary_interval TEXT,
            salary_min_amount REAL,
            salary_max_amount REAL,
            salary_currency TEXT,
            date_posted TIMESTAMP,
            is_remote BOOLEAN,
            job_function TEXT,
            company_industry TEXT,
            source_site TEXT,
            scrape_date TIMESTAMP,
            UNIQUE(job_url, company, title)
        )
        """)
        
        conn.commit()
        conn.close()

    def generate_job_id(self, row: Dict[str, Any]) -> str:
        """Generate a unique job ID based on job details."""
        components = [
            str(row.get('job_url', '')),
            str(row.get('company', '')),
            str(row.get('title', '')),
            str(row.get('location_city', '')),
            str(row.get('date_posted', ''))
        ]
        return '_'.join(components)

    def transform_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """Transform and clean the scraped data."""
        if df is None or df.empty:
            return pd.DataFrame()
            
        # Generate unique job IDs
        df['job_id'] = df.apply(self.generate_job_id, axis=1)
        
        # Clean salary data
        df['salary_min_amount'] = pd.to_numeric(df['min_amount'], errors='coerce')
        df['salary_max_amount'] = pd.to_numeric(df['max_amount'], errors='coerce')
        df['salary_interval'] = df['interval']
        
        # Clean and standardize columns
        df['date_posted'] = pd.to_datetime(df['date_posted'], errors='coerce')
        df['scrape_date'] = pd.to_datetime(df['scrape_date'])
        
        # Select and rename columns for database
        columns_mapping = {
            'job_id': 'job_id',
            'title': 'title',
            'company': 'company',
            'company_url': 'company_url',
            'job_url': 'job_url',
            'country': 'location_country',
            'city': 'location_city',
            'state': 'location_state',
            'description': 'description',
            'job_type': 'job_type',
            'salary_interval': 'salary_interval',
            'salary_min_amount': 'salary_min_amount',
            'salary_max_amount': 'salary_max_amount',
            'currency': 'salary_currency',
            'date_posted': 'date_posted',
            'is_remote': 'is_remote',
            'job_function': 'job_function',
            'company_industry': 'company_industry',
            'source_site': 'source_site',
            'scrape_date': 'scrape_date'
        }
        
        return df.rename(columns=columns_mapping)[list(columns_mapping.values())]

    def deduplicate_jobs(self, df: pd.DataFrame) -> pd.DataFrame:
        """Remove duplicate jobs and check against existing database entries."""
        if df.empty:
            return df

        # Remove duplicates within the current DataFrame
        df = df.drop_duplicates(subset=['job_url', 'company', 'title'])

        # Check for existing entries in the database
        conn = sqlite3.connect(self.db_path)
        existing_jobs = pd.read_sql("""
            SELECT job_url, company, title 
            FROM jobs
        """, conn)
        conn.close()

        if not existing_jobs.empty:
            # Merge with existing jobs and keep only new ones
            merged = df.merge(
                existing_jobs,
                on=['job_url', 'company', 'title'],
                how='left',
                indicator=True
            )
            df = merged[merged['_merge'] == 'left_only'].drop('_merge', axis=1)

        return df

    def load_to_database(self, df: pd.DataFrame):
        """Load transformed and deduplicated data into SQLite database."""
        if df.empty:
            self.logger.warning("No data to load into database")
            return
            
        try:
            # Deduplicate before loading
            df = self.deduplicate_jobs(df)
            
            if df.empty:
                self.logger.info("No new unique jobs to load after deduplication")
                return
            
            # Load to SQLite
            conn = sqlite3.connect(self.db_path)
            df.to_sql('jobs', conn, if_exists='append', index=False, 
                     method='multi', chunksize=1000)
            conn.commit()
            conn.close()
            
            self.logger.info(f"Successfully loaded {len(df)} unique jobs into database")
            
        except Exception as e:
            self.logger.error(f"Error loading data to database: {str(e)}")

    def process_location_title_pair(self, location: str, job_title: str) -> int:
        """Process a single location and job title pair."""
        jobs_processed = 0
        sites = ["indeed", "linkedin", "zip_recruiter", "glassdoor"]
        
        for site in sites:
            try:
                # Add random delay between searches to avoid rate limiting
                time.sleep(random.uniform(1, 3))
                
                # Extract
                raw_data = self.scrape_jobs_for_location(location, job_title, site)
                
                if raw_data is not None and not raw_data.empty:
                    # Transform
                    transformed_data = self.transform_data(raw_data)
                    
                    if not transformed_data.empty:
                        # Load
                        self.load_to_database(transformed_data)
                        jobs_processed += len(transformed_data)
                    
            except Exception as e:
                self.logger.error(f"Pipeline error for {job_title} in {location} from {site}: {str(e)}")
                continue
        
        return jobs_processed

    def run_etl_pipeline(self, max_workers: int = 4):
        """Execute the complete ETL pipeline with parallel processing."""
        total_jobs = 0
        job_pairs = [(loc, title) for loc in self.search_locations for title in self.job_titles]
        
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_pair = {
                executor.submit(self.process_location_title_pair, loc, title): (loc, title) 
                for loc, title in job_pairs
            }
            
            for future in concurrent.futures.as_completed(future_to_pair):
                loc, title = future_to_pair[future]
                try:
                    jobs_processed = future.result()
                    total_jobs += jobs_processed
                except Exception as e:
                    self.logger.error(f"Pipeline error for {title} in {loc}: {str(e)}")
        
        self.logger.info(f"ETL pipeline completed. Total jobs processed: {total_jobs}")
        
        # Run analytics after completion
        self.run_analytics()

    def run_analytics(self):
        """Run analytics queries on the collected data using SQLite and pandas."""
        try:
            conn = sqlite3.connect(self.db_path)
            
            # Salary analytics
            salary_analytics = pd.read_sql("""
                SELECT 
                    location_city,
                    job_type,
                    COUNT(*) as job_count,
                    AVG(salary_min_amount) as avg_min_salary,
                    AVG(salary_max_amount) as avg_max_salary,
                    MIN(salary_min_amount) as min_salary,
                    MAX(salary_max_amount) as max_salary
                FROM jobs
                WHERE salary_min_amount IS NOT NULL
                GROUP BY location_city, job_type
            """, conn)
            
            # Company analytics
            company_analytics = pd.read_sql("""
                SELECT 
                    company,
                    COUNT(*) as total_jobs,
                    COUNT(DISTINCT location_city) as locations,
                    COUNT(DISTINCT job_type) as job_types,
                    AVG(CASE WHEN salary_min_amount IS NOT NULL THEN salary_min_amount END) as avg_min_salary
                FROM jobs
                GROUP BY company
            """, conn)
            
            conn.close()
            
            # Save to CSV
            salary_analytics.to_csv('salary_analytics.csv', index=False)
            company_analytics.to_csv('company_analytics.csv', index=False)
            
            self.logger.info("Analytics completed and exported to CSV files")
            
        except Exception as e:
            self.logger.error(f"Error running analytics: {str(e)}")

if __name__ == "__main__":
    # Initialize and run the ETL pipeline
    etl = JobScraperETL()
    etl.run_etl_pipeline(max_workers=4)

2025-02-02 00:24:36,556 - INFO - Scraping jobs for software engineer in New York, NY from indeed
2025-02-02 00:24:37,113 - INFO - Scraping jobs for data scientist in New York, NY from indeed
2025-02-02 00:24:37,236 - INFO - Scraping jobs for software developer in New York, NY from indeed
2025-02-02 00:24:37,965 - INFO - Scraping jobs for machine learning engineer in New York, NY from indeed
2025-02-02 00:24:38,129 - ERROR - Pipeline error for software engineer in New York, NY from indeed: "['location_country', 'location_city', 'location_state'] not in index"
2025-02-02 00:24:38,179 - ERROR - Pipeline error for data scientist in New York, NY from indeed: "['location_country', 'location_city', 'location_state'] not in index"
2025-02-02 00:24:39,425 - INFO - Scraping jobs for software engineer in New York, NY from linkedin
2025-02-02 00:24:39,470 - ERROR - Pipeline error for machine learning engineer in New York, NY from indeed: "['location_country', 'location_city', 'location_state'] not